<h1>Predicting Heart Disease with Naive Bayes</h1>

The World Health Organization has estimated that 12 million deaths occur worldwide every year due to heart disease. Half the deaths in the United States and other developed countries are due to cardiovascular disease. Early prognosis of cardiovascular disease can aid decisions on lifestyle changes for high-risk patients, and in turn reduce complications. In this lab we identify the most relevant risk factors of heart disease and predict overall 10-year risk using a <b>Naive Bayes classifier</b>.

<h3>What is Naive Bayes?</h3>

Naive Bayes is a classification algorithm based on Bayes' theorem. It calculates the probability of each class of coronary heart disease (CHD or no CHD) given the patient's measurements, and predicts the more probable class. It is called <i>naive</i> because it assumes every feature contributes **independently** to the outcome. Real features are rarely fully independent, but the algorithm is fast, simple, and often works surprisingly well. We use <b>GaussianNB</b>, the variant for continuous measurements.

<h3>The dataset</h3>

The dataset is publicly available on Kaggle (https://www.kaggle.com) and comes from an ongoing cardiovascular study of residents of Framingham, Massachusetts. The classification goal is to predict whether a patient has a 10-year risk of future coronary heart disease (CHD). It includes over 4,000 records and 15 attributes.

<u>Demographic</u>
<ul>
    <li>`sex_male`: male or female (nominal)</li>
    <li>`age`: age of the patient (continuous)</li>
    <li>`education`: education level (dropped in this lab)</li>
</ul>

<u>Behavioural</u>
 <ul>   
    <li>`currentSmoker`: whether the patient is a current smoker (nominal)</li>
    <li>`cigsPerDay`: average number of cigarettes smoked per day (continuous)</li>
</ul>


<u>Medical (history)</u>
 <ul>
    <li>`BPMeds`: on blood pressure medication (nominal)</li>
    <li>`prevalentStroke`: previously had a stroke (nominal)</li>
    <li>`prevalentHyp`: hypertensive (nominal)</li>
    <li>`diabetes`: has diabetes (nominal)</li>
</ul>


<u>Medical (current)</u>
<ul>    
    <li>`totChol`: total cholesterol (continuous)</li>
    <li>`sysBP`: systolic blood pressure (continuous)</li>
    <li>`diaBP`: diastolic blood pressure (continuous)</li>
    <li>`BMI`: body mass index (continuous)</li>
    <li>`heartRate`: heart rate (continuous)</li>
    <li>`glucose`: glucose level (continuous)</li>
</ul>

<u>Target</u>
 <ul>   
    <li>`TenYearCHD`: 10-year risk of coronary heart disease ("1" = yes, "0" = no)</li>
</ul>

In [ ]:
# import the libraries we need:
# pandas      -> loading and working with the data table
# numpy       -> number handling and array operations
# matplotlib  -> plotting
# seaborn     -> the correlation heatmap and confusion-matrix heatmap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# read the Framingham heart study data - 4,238 patients
df = pd.read_csv('data/framingham.csv')
print(df.isnull().sum())                                # how many missing values in each column?
df.dropna(axis=0, inplace=True)                         # drop rows with any missing value
df.drop(['education'], axis=1, inplace=True)            # keep the feature set to clinical and behavioural measures
df.rename(columns={'male':'sex_male'}, inplace=True)    # clearer column name
print(df.shape)                                         # rows and columns after cleaning: (3656, 15)
df.head()                                               # first look at the cleaned data


In [ ]:
# one styling call: whitegrid background, notebook sizing, larger fonts
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.5)

# the 14 feature columns we will check for correlation
cols = ['sex_male','age', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 
        'diabetes','totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate','glucose']

# correlation between every pair of features (a 14 x 14 grid of values)
cm = np.corrcoef(df[cols].values.T)

# draw it as a heatmap: annot=True prints each value, fmt='.2f' = 2 decimal places
f, ax = plt.subplots(figsize=(15,10))
heatMap = sns.heatmap(cm, cbar=True, annot=True,square=True, fmt='.2f', annot_kws={'size' : 15}, yticklabels=cols, 
xticklabels=cols)
plt.show()


<h3>Removing highly correlated features</h3>

Naive Bayes assumes every feature is independent. When two features are strongly correlated they carry almost the same information, and the model effectively counts that evidence twice. The function below scans the correlation matrix and flags any feature with a correlation above 0.7 with an earlier feature, so we can drop it.

In [ ]:
# scan the correlation matrix and collect the name of any feature whose
# correlation with an earlier feature is stronger than the threshold -
# those are the columns we will drop before fitting Naive Bayes
def correlation(dataset, threshold):
    col_corr = set()                                    # Set of all the names of correlated columns
    corr_matrix = dataset.corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) > threshold: # we are interested in absolute coeff value
                colname = corr_matrix.columns[i]        # getting the name of column
                col_corr.add(colname)
    return col_corr

# run it on our 14 features with a 0.7 threshold and show what it found
corr_features = correlation(df[cols], 0.7)
corr_features




In [ ]:
# drop cigsPerDay and diaBP — each is highly correlated with a feature we keep
# (cigsPerDay with currentSmoker, diaBP with sysBP)
df.drop(corr_features,axis=1, inplace=True)
df.head()


In [ ]:
# x = every column except the last (the 12 features)
# y = the last column only (TenYearCHD — the value we want to predict)
# .values converts from a DataFrame to a plain NumPy array
x = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# separate dataset into train (70%) and test (30%)
# random_state=0 makes the split reproducible — same rows every run
# stratify=y keeps the CHD rate (~15%) the same in both sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=0, stratify=y)

# confirm the split: (rows, features) for each set
X_train.shape, X_test.shape


In [ ]:
# standardise the features: rescale each column to mean 0, standard deviation 1,
# so big-number features (e.g. totChol ~240) don't drown out small ones (e.g. sex_male 0/1)
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

# fit_transform on TRAIN: learn each column's mean and std, then rescale
X_train = sc.fit_transform(X_train)

# transform only on TEST: reuse the means and stds learned from the training set
# (never fit on the test set — that would leak test information into the model)
X_test = sc.transform(X_test)

In [ ]:
# create the Gaussian Naive Bayes classifier
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()

# fit = learn from the training data. For each class (CHD / no CHD) the model learns
# how common the class is, plus each feature's mean and spread (a bell curve per feature).
# To predict, it asks: which class makes this patient's numbers most probable?
classifier.fit(X_train, y_train)

In [ ]:
# ask the fitted model to predict CHD (1) or no CHD (0) for every test patient
y_pred = classifier.predict(X_test)

# show the predictions and the actual answers side by side:
# left column = predicted, right column = actual
# (reshape turns each 1-D array into a column; concatenate glues the columns together)
print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

In [ ]:
# count how the predictions line up with the actual answers
from sklearn.metrics import confusion_matrix, accuracy_score

# rows = actual, columns = predicted:
# [[ true negatives    false positives ]
#  [ false negatives   true positives  ]]
cm = confusion_matrix(y_test, y_pred)
print(cm)

# accuracy = (true negatives + true positives) / all test patients
accuracy_score(y_test, y_pred)

In [ ]:

# rebuild the confusion matrix (same numbers as the previous cell)
cmatrix = confusion_matrix(y_test, y_pred)

# wrap it in a DataFrame so the rows and columns carry readable labels
conf_matrix = pd.DataFrame(data=cmatrix, columns=['Predicted:0','Predicted:1'], index=['Actual:0','Actual:1'])

# draw it as a heatmap: annot=True prints the count inside each square,
# fmt='d' shows the counts as whole numbers, cmap is the colour scale (yellow = low, red = high)
plt.figure(figsize = (10,8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='YlOrRd')


<h3>Beyond accuracy</h3>

Accuracy looks respectable here — but look closer at the confusion matrix.
The precision, recall and f1-score below tell the real story for the patients who actually developed CHD.

In [ ]:
# the classification report scores each class separately:
#   precision = of the patients the model flagged as this class, how many really were
#   recall    = of the patients who really ARE this class, how many the model caught
#   f1-score  = the balance of precision and recall (their harmonic mean)
#   support   = how many test patients are actually in this class
from sklearn.metrics import classification_report

# target_names swaps the 0/1 labels for readable ones in the printout
print(classification_report(y_test, y_pred, target_names=['No CHD (0)', 'CHD (1)']))

<h1>What did we learn?</h1>
<ul>
    <li><u>Check before you clean:</u> <b>isull().sum()</b> showed glucose alone had 388 missing values. Dropping every incomplete row cost us 582 of 4,238 records, leaving 3,656.</li>
    <li><u>Naive Bayes assumes independent features:</u> Two pairs were highly correlated — cigsPerDay with currentSmoker (0.77) and diaBP with sysBP (0.79) — so we dropped one of each pair rather than let the model count the same evidence twice.</li>
    <li><u>Stratifying the split matters on imbalanced data:</u> Only ~15% of patients developed CHD, so <b>stratify=y</b> keeps that rate the same in the training and test sets.</li>
    <li><u>Accuracy said 83% — and it was misleading:</u> The test set is 85% CHD-negative, so a "model" that predicts *nobody* gets heart disease scores 84.8% and beats our classifier without learning anything.</li>
    <li><u>The confusion matrix and classification report told the real story:</u> The model caught only 30 of the 167 patients who actually developed CHD (recall 0.18) and missed 137 — for a screening tool, the most dangerous kind of error.</li>

<b>The lesson:</b> on imbalanced data, never judge a model by accuracy alone. Precision, recall and f1 per class show what accuracy hides.